# Train a model on Colab

Runs `python -m src.train` on a Colab GPU with the data, checkpoints and results on Google Drive,
so a disconnect loses nothing: run all cells again and training resumes from the last checkpoint.

Drive must contain (see `DRIVE_ROOT` below):
```
who_wrote_this/data/splits/train.parquet
who_wrote_this/data/splits/val.parquet
who_wrote_this/results/class_weights.json
```

In [ ]:
# Settings
MODEL = "transformer"            # transformer | lstm | cnn | mlp
BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/who_wrote_this"
REPO_URL = "https://github.com/VEDITHREDDY26/Who_Wrote_this.git"
REPO_DIR = "/content/Who_Wrote_this"

In [ ]:
# 1. GPU check (Runtime > Change runtime type > A100 / L4 / T4)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 3. Code: clone once, then always update to the latest commit of BRANCH
import os
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git fetch -q origin && git checkout -q {BRANCH} && git reset -q --hard origin/{BRANCH}
!git log --oneline -1

In [ ]:
# 4. Packages (Colab already has torch with CUDA)
!pip install -q -r requirements.txt

In [ ]:
# 5. Paths on Drive, then check that the data is there
from pathlib import Path
os.environ["TD_DATA_DIR"] = f"{DRIVE_ROOT}/data"
os.environ["TD_RESULTS_DIR"] = f"{DRIVE_ROOT}/results"
os.environ["TD_CHECKPOINT_DIR"] = f"{DRIVE_ROOT}/checkpoints"
needed = [f"{DRIVE_ROOT}/data/splits/train.parquet", f"{DRIVE_ROOT}/data/splits/val.parquet",
          f"{DRIVE_ROOT}/results/class_weights.json"]
missing = [p for p in needed if not Path(p).exists()]
if missing:
    raise FileNotFoundError(f"Upload these to Google Drive first: {missing}")
!python scripts/check_env.py
!python -c "from src.data.splits import load_split, fingerprint_ids; [print(s, len(d), fingerprint_ids(d['id'])) for s in ('train', 'val') for d in [load_split(s, columns=['id'])]]"
# expected: train 153459 012a901cb6375b61 / val 17101 6c517d46f22dda76

In [ ]:
# 6. (Optional) quick check on 2,000 rows, about 2-3 minutes
!python -m src.train --model {MODEL} --subset 2000 --max-epochs 1 --run-name colab-smoke

In [ ]:
# 7. Full training. If Colab disconnects, run cells 2-5 and this cell again: it resumes.
!python -m src.train --model {MODEL}

In [ ]:
# 8. Results
import json
m = json.load(open(f"{DRIVE_ROOT}/results/{MODEL}_val_metrics.json"))
print(f"val macro-F1 {m['macro_f1']:.4f}  accuracy {m['accuracy']:.4f}  human FPR {m['human_fpr']:.4f}")
print(f"train time {m['extra']['train_time_seconds'] / 60:.1f} min, {m['extra']['num_parameters']:,} parameters")
for name, pc in m["per_class"].items():
    print(f"  {name:8s} P {pc['precision']:.3f}  R {pc['recall']:.3f}  F1 {pc['f1']:.3f}")